# Day 7: Incremental Processing & CDC Upsert Engine

This notebook demonstrates Idempotent UPSERT/DELETE logic on a simulated stateful table using Iceberg MERGE INTO.

In [ ]:
from pyspark.sql import SparkSession
import os

warehouse_path = os.path.join(os.getcwd(), "spark-warehouse")

spark = SparkSession.builder \
    .appName("Day07_Incremental_Merge") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", warehouse_path) \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session created.")

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.ehdip_silver")

# Create base target table
spark.sql("""
CREATE TABLE IF NOT EXISTS local.ehdip_silver.omop_person (
    person_id INT,
    gender_concept_id INT,
    year_of_birth INT,
    updated_at TIMESTAMP
)
USING iceberg
TBLPROPERTIES (
    'format-version'='2',
    'write.update.mode'='merge-on-read',
    'write.delete.mode'='merge-on-read',
    'write.merge.mode'='merge-on-read'
)
""")

# Insert initial state
spark.sql("""
INSERT INTO local.ehdip_silver.omop_person VALUES 
(1, 8507, 1980, timestamp '2023-01-01 10:00:00'),
(2, 8532, 1990, timestamp '2023-01-01 10:00:00')
""")
print("Initial State:")
spark.sql("SELECT * FROM local.ehdip_silver.omop_person").show()

In [ ]:
# Simulate incoming CDC updates (op 'u' for update, 'd' for delete, 'c' for create)
# Includes a late-arriving update that should be ignored because its updated_at is older
updates_data = [
    (1, 8507, 1981, 'u', '2023-01-02 10:00:00'), # Valid update
    (2, 8532, 1990, 'd', '2023-01-02 11:00:00'), # Valid delete
    (3, 8507, 2000, 'c', '2023-01-02 12:00:00'), # Valid insert
    (1, 8507, 1979, 'u', '2022-12-31 10:00:00')  # Late arriving out-of-order update, should ignore
]
updates_df = spark.createDataFrame(updates_data, ["person_id", "gender_concept_id", "year_of_birth", "op", "updated_at"])
updates_df.createOrReplaceTempView("updates")

merge_sql = """
MERGE INTO local.ehdip_silver.omop_person t
USING updates u
ON t.person_id = u.person_id
WHEN MATCHED AND u.op = 'd' THEN DELETE
WHEN MATCHED AND u.updated_at > t.updated_at THEN UPDATE SET *
WHEN NOT MATCHED AND u.op != 'd' THEN 
    INSERT (person_id, gender_concept_id, year_of_birth, updated_at) 
    VALUES (u.person_id, u.gender_concept_id, u.year_of_birth, u.updated_at)
"""

spark.sql(merge_sql)
print("State after UPSERT/DELETE logic (Person 2 deleted, 3 inserted, 1 updated to 1981, late update ignored):")
spark.sql("SELECT * FROM local.ehdip_silver.omop_person ORDER BY person_id").show()